# FED Rates Policy Decision Analyzer
### Two-step logistic regression with rolling window


## 1 - Imports

In [ ]:
import pandas as pd
import numpy as np
from pandas_datareader import data as pdr
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score, roc_auc_score)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

## 2 - Data collection and cleaning

In [ ]:

START_DATE = "1990-01-01"
END_DATE = datetime.today().strftime('%Y-%m-%d')

# FRED series codes
FRED_SERIES = {
    "fed_rate"  : "FEDFUNDS", # Federal Funds Rate
    "cpi"       : "CPIAUCSL", # CPI
    "core_cpi"  : "CPILFESL", # Core CPI
    "unrate"    : "UNRATE", # Unemployment rate
    "indpro"    : "INDPRO", # Industrial production index
    "wages"     : "AHETPI", # Average hourly earnings
    "dgs2"      : "DGS2", # 2 year Treasury yield
    "dgs10"     : "DGS10", # 10 year Treasury yield
    "vix"       : "VIXCLS", # VIX volatility index
}

blank_frame = []
for col_name, fred_code in FRED_SERIES.items():
    tmp = pdr.DataReader(fred_code, 'fred', START_DATE, END_DATE)
    tmp.rename(columns={fred_code: col_name}, inplace=True)
    blank_frame.append(tmp)

df_init = pd.concat(blank_frame, axis=1)

# Resample to monthly frequency
# 1) forward-fill --> 2) take the monthly mean
# N.B. : use the mean rather than the last observation gives a more representative picture of the month -- at the cost of smoothing intra-month volatility
df_monthly = df_init.ffill().resample('MS').mean().dropna()

## 3 - Feature engineering and target construction

In [ ]:
# Financial variables
df_monthly['yield_slope'] = df_monthly['dgs10'] - df_monthly['dgs2']   # Yield slope := spread 10y - 2y bond yields
df_monthly['log_vix'] = np.log(df_monthly['vix']) # logarithm of VIX

# Macroeconomic variables
df_monthly['core_infl_yoy'] = df_monthly['core_cpi'].pct_change(12) * 100 # Core inflation YoY
df_monthly['ip_growth_yoy'] = df_monthly['indpro'].pct_change(12) * 100   # Industrial production growth YoY
df_monthly['wage_growth'] = (df_monthly['wages'].pct_change(12) * 100).diff()  # change in wage growth

# Unemployment gap: deviation from 24-month rolling mean.
df_monthly['unrate_gap'] = df_monthly['unrate'] - df_monthly['unrate'].rolling(24).mean()


# 2) Construction of depedent variable


# Monthly change in the Fed Funds Rate
df_monthly['rate_change'] = df_monthly['fed_rate'].diff()

# Classify each month into three outcomes:
#   0 = Hold  (no meaningful rate move)
#   1 = Hike  (rate increase >= 12.5 bps)
#   2 = Cut   (rate decrease >= 12.5 bps)
# N.B. : the arbitrary 12.5 bps threshold filters out rounding noise in the FRED monthly average
df_monthly['policy_move'] = 0
df_monthly.loc[df_monthly['rate_change'] >  0.125, 'policy_move'] = 1
df_monthly.loc[df_monthly['rate_change'] < -0.125, 'policy_move'] = 2

## 4 - Identification strategy — introduction of lags

In [ ]:

# We only use information that would realistically be available to the Fed at the time of decision, that is 
# variables known the month before the decision 

# Data availability varies depending on considered variable. We distinguish : 
# "fast" variables (financial market data): available within or with a 1-month lag
# "slow" variables (macro releases): available with a 2-month lag, due to publication delay

Instant_vars = ['yield_slope', 'log_vix', 'policy_move']
Delay_vars = ['core_infl_yoy', 'ip_growth_yoy', 'unrate_gap', 'wage_growth']

for col in Instant_vars:
    df_monthly[f'{col}_lag1'] = df_monthly[col].shift(1)

for col in Delay_vars:
    df_monthly[f'{col}_lag2'] = df_monthly[col].shift(2)

# Final data frame that will be used by the prediction algorithm : df_final
df_final = df_monthly.dropna().copy()



# Set a binary indicator: did the Fed act (hike or cut) in a given month
df_final['acted'] = (df_final['policy_move'] != 0).astype(int)

# Feature list used in both steps of the model
FEATURES = (
    [f'{v}_lag1' for v in Instant_vars] +
    [f'{v}_lag2' for v in Delay_vars]
)

# Note : in Instant_vars, we include policy_move_lagged
# This specification decision is not neutral. It is supported by the empirical observation that there exists an intertia in FED monetary policy
# While it can introduce endogeneity and auto correlation, it is intended to capture monatary policy persistance besides pure macroeconomic and financial data

## 5 - Final dataset and feature verification (ADF and VIF)

In [ ]:


# Stationarity test with ADF (p-value at 5% threshold)
def check_stationarity(df, cols):
    for col in cols:
        p_val = adfuller(df[col].dropna())[1]
        status = "Stationary" if p_val < 0.05 else "Non stationary"
        print("Variable", col, "p-value =", p_val, "Result", status)

# Multicollinearity check among features with VIF
def check_vif(df, cols):
    X = df[cols].dropna() 
    vif_df = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
    })
    return vif_df

check_stationarity(df_final, FEATURES)
vif_results = check_vif(df_final, FEATURES)
print(vif_results)

## 6 - Main function — two-step rolling-window logit


In [ ]:
''' Identification strategy (two steps) : 
- Step 1) Will the Fed act?   (Hold vs. Pivot, binary logit with dependent variable `acted`)
- Step 2) If so, which way?   (Hike vs. Cut, conditional binary logit - on "acted" - with dependent variable `policy_move`) 

The rolling window ensures the model is always trained only on past data : possible and genuine out-of-sample evaluation at each point in time  -->  avoid look-ahead biais'''


def two_step_logit(df, features, window):
    predictions = [] # Store monthly estimations throughout the loop
    coef_full_record = []
    
    # Definition of the rolling dynamic

    # We have the time window that defines how much months will be considered in the first iteration
    # In this first iteration : the function takes "window" observations from the data frame to train (df.iloc[:window]) 
    # and tests the estimations on the very next observation df.iloc[window:window+1] 

    # Then, we enter the loop that goes from window to the very last monthly observation in the data frame (the full length of df)

    # At step i : the function takes all observations in df until month i included to train, and tests it on the next monthly observation i+1

    for i in range(window, len(df)): 
        train = df.iloc[:i]
        test  = df.iloc[i:i+1]
        date  = test.index[0] # At each step in the loop, we note the exact month used as test

        # Step 1: Pivot vs Hold --> m1
        m1 = LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs')
        # Note : class_weight = 'balanced' : the data frame registers many "holds" with the 12.5 bps threshold, an imbalance to be checked and balanced 
        # when training and estimating

        # First logit
        # Arguments : train the link between features and the variable "acted" (FED pivoted, not "hold") - defined above as policy_move ≠ 0
        m1.fit(train[features], train['acted']) 
        # Store coefficients to track how sensitivities evolve over time
        coef_record = dict(zip(features, m1.coef_[0])) 
        coef_record['date'] = date
        coef_full_record.append(coef_record)

        # Step 2: Hike vs Cut (conditional on a pivot being predicted) --> m2
        train_pivot = train[train['policy_move'] != 0] # Only look at months in "train" where the FED "acted"
        m2 = LogisticRegression(max_iter=1000, solver='lbfgs') 
        # Note : no balanced weights. In this model, we shall assume that the unbalance between hikes and cuts does not justify it (opposed to the m1 regression) 
        m2.fit(train_pivot[features], train_pivot['policy_move']) # Train the model to distinguish policy moves (hikes and cuts)
        # Gives an array, 

        # Prediction (on the "test" month i+1)
        step1_pred  = m1.predict(test[features])[0] # Prediction : pivot (1) or hold (0)
        prob_pivot = m1.predict_proba(test[features])[0][1] # Extract the conviction of the prediction on test month movement
        # [0][1] --> we take the second element (probability to pivot) of the list prob_pivot produced by m1.predict

        dir_probs    = m2.predict_proba(test[features])[0] # Given that there is a pivot (1), what are the probability for each direction (hike or cut)
        # Output : an array with probability to hike and cut

        classes = m2.classes_



        # Now, we want to extract the infotmation in the first row [0] of dir_pivot : hike et pivot
        prob_cut_cond = dir_probs[np.where(classes == 2)[0][0]] # Look where in dir_probs there was a cut
        prob_hike_cond = dir_probs[np.where(classes == 1)[0][0]] # Look where in dir_probs there was a hike

        final_pred = 0 if step1_pred == 0 else m2.predict(test[features])[0]
        # If prediction for period i+1 states 0 (hold) : final prediction is equal to 0
        # If prediction for period i+1 states 1 (pivot) : compute prediction of whether hike (1) or cut (2)

        # For month i+1 : we store everything (date of considered test month, actual empirical observation, prediction, probabilities)
        predictions.append({
            'date' : date,
            'actual' : test['policy_move'].values[0],
            'predicted' : final_pred,
            'prob_pivot' : prob_pivot,
            'prob_hike_total': prob_pivot * prob_hike_cond,
            'prob_cut_total' : prob_pivot * prob_cut_cond,
        })

    df_preds = pd.DataFrame(predictions).set_index('date') # Data frame to store predictions at the end of the loop, indexed by date
    df_coefs = pd.DataFrame(coef_full_record).set_index('date') # Data frame to store estimated weight of each feature in FED's decision, indexed by date

    return df_preds, df_coefs


# Run with a 72-month (6-year) initial training window
WINDOW = 72

# Run the mdodel
df_preds, df_coefs = two_step_logit(df_final, FEATURES, WINDOW)


print(df_coefs)
print(df_preds)

## 7 - Model evaluation — out of sample

In [ ]:


effective_preds_df = df_preds.dropna(subset=['predicted', 'actual']) # We do not keep lines in df_preds where there is either (or both) perdiction or actual value missing
y_true = effective_preds_df['actual'].astype(int) # Integer of the extracted effective, true value 
y_pred = effective_preds_df['predicted'].astype(int) # Integer of the predicred value

print(f"  Accuracy: {accuracy_score(y_true, y_pred)}") # Accuracy score of 0,1 or 2 prediction vs actual, from numpy package

print("Classification Report (0: Hold, 1: Hike, 2: Cut)")
print(classification_report(y_true, y_pred)) # Classification report (precision, recall metrics) from sklearn.metrics : 0 vs 1 vs 2

# AUC for the pivot detection step (binary: act vs hold)
y_true_pivot = (y_true != 0).astype(int) # Hold remains 0 and hike and cut both become 1 --> detect policy movement

# Compute : for an aleatory pair (observed vs predicted pivot) does the model assign higher pivot probability given that 
# Evaluate relevance of step 1 (m1 computation) and estimated probabilities
auc_pivot = roc_auc_score(y_true_pivot, effective_preds_df['prob_pivot']) # ROC AUC with sklearn.metrics
print(f" ROC AUC (Pivot vs Hold): {auc_pivot}")

# Precision when model signals a hike or cut (conditional on "act" not hold)
'''
                Pred Hold       Pred Hike       Pred Cut
Real Hold        [0,0] Correct                     
Real Hike                       [1,1] Correct     
Real Cut                                        [2,2] Correct
'''
cm = confusion_matrix(y_true, y_pred) # Confusion matrix from sklearn.metrics
hike_precision = cm[1,1] / sum(cm[:,1]) # cm[:,1] : the whole Pred Hike column (model predicted Hike)
cut_precision = cm[2,2] / sum(cm[:,2]) # cm[:,2] : the whole Pred Cut column (model predicted Cut)
print(f"  Precision when predicting Hike : {hike_precision:.2%}")
print(f"  Precision when predicting Cut  : {cut_precision:.2%}")

# Confusion matrix plot
cm_disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Hold', 'Hike', 'Cut'])
cm_disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Two-Step Rolling Logit')
plt.tight_layout()
plt.show()

## 8 · Window optimisation

A tool to fix parameter `WINDOW` in the main two step logit function.

In [ ]:


def optimise_window(df, features, window_range):
    track_record = []
    for w in window_range:
        res, _ = two_step_logit(df, features, w) # Note that we only keep df_preds from two_step_logit in res, not df_coefs (coefficients not useful here)
        ev = res.dropna(subset=['predicted', 'actual']) # Actual data frame to study
        yt = ev['actual'].astype(int)
        yp = ev['predicted'].astype(int)

        acc = accuracy_score(yt, yp)
        auc = roc_auc_score((yt != 0).astype(int), ev['prob_pivot'])

        track_record.append({'window': w, 'accuracy': acc, 'auc_pivot': auc, 'n_obs': len(ev)})
        print(f"  Window {w} / Acc: {acc}  / AUC: {auc}")

    return pd.DataFrame(track_record)

# N.B. : the window optimization covers *all* the data at once, unlike the rolling logit (based on this very window)
# Possible look ahead biais




# Test windows from 30 to 144 months in steps of 6
window_pace = range(30, 145, 6)
df_opt = optimise_window(df_final, FEATURES, window_pace)
best_window = df_opt.loc[df_opt['accuracy'].idxmax(), 'window']
# df_opt['accuracy'].idxmax() --> maximize computed accuracy
# Because we have indexed by window, we extract this information with df_opt.loc
print(f"Best window by accuracy: {best_window} months")

## 9 - Graphic confrontation — predicted signals vs observed rate

In [ ]:
# 8) Graphic confrontation of double Logit prediction vs observed rates

# At this point, the model decides 1) if the FED will hold or pivot, with a given probability, and 2) given this pivot probability, whether this pivot will be a hike or cut
# We use this pivot probability to determine the conviction of the model that the FED will pivot in one or the other direction
# We build a direction probability variable dir_prob, that tells each month what the model predicts and its degree of conviction

df_preds['dir_prob'] = df_preds.apply(
    lambda r:  r['prob_pivot'] if r['predicted'] == 1 # If 1 (a hike) us predicted by the model : we take the estimated pivot probaiblity, and assign it a positive value
    else  -r['prob_pivot'] if r['predicted'] == 2 # If 2 (a cut) is predicted by the model : we take the estimated pivot probability, and assign it a negative value
    else 0, # Hold, null pivot direction probability
    axis=1 # Probability : axis goes from -1 (negative assigned probaiblity) to 1
)




fig, ax1 = plt.subplots(figsize=(14, 7))

# Shaded area curve : model's conviction of FED's estimated move in each direction

# 1) Shaded area in red if dir_prob>0 (Hike)
ax1.fill_between(df_preds.index, 0, df_preds['dir_prob'],
                 where=(df_preds['dir_prob'] > 0),
                 color='tab:red', alpha=0.2, label='Hike probability')
# 2) Shaded area in green if dir_prob<0 (Cut)
ax1.fill_between(df_preds.index, 0, df_preds['dir_prob'],
                 where=(df_preds['dir_prob'] < 0),
                 color='tab:green', alpha=0.2, label='Cut probability')

# Rolling coefficients: how the model's sensitivity to each variable (core inflation and unemployment, dual mandate features) evolves in time
# How determinant are core inflation and unemployment in the model's prediction decision
# Use dataframe df_coefs from the two_step_logit function
ax1.plot(df_coefs.index, df_coefs['core_infl_yoy_lag2'],
         color='tab:blue',   lw=1.5, alpha=1, label='Core inflation')
ax1.plot(df_coefs.index, df_coefs['unrate_gap_lag2'],
         color='tab:orange', lw=1.5, alpha=1, label='Unemployment gap')

# Horizontal line at y=0 for reference
ax1.axhline(0, color='black', lw=1, alpha=0.5)
ax1.set_ylabel('Model estimation : pivot probability and coefficient sensitivity)')
ax1.set_ylim(-1.5, 1.5) # Left axis ax.1 : range for probabilites and sensitivities, that will be set within [-1.5 ; 1.5]

# New layer, superposed to the elements above
# Right axis ax.2 : actual Fed Funds Rate (solid black curve)
ax2 = ax1.twinx()
ax2.plot(df_final.index, df_final['fed_rate'],
         color='black', lw=2.5, alpha=0.9, label='Fed Fund Rate (observed)')
ax2.set_ylabel('Fed Funds Rate (%)')
ax2.set_ylim(0, 9) # Fix observed Fed Fund Rates between 0% and 9%


ax1.legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)

ax1.grid(alpha=0.1)
plt.title('Fed Policy Model — Predicted Signals vs Realised Rate', fontsize=12)
plt.tight_layout()
plt.show()